# 06-03 泛化能力与训练曲线诊断：模型到底有没有真的学会

前面我们一直在学习怎么训练神经网络：

```text
前向传播 -> 损失函数 -> 反向传播 -> 优化器 -> 学习率调整
```

这些内容主要解决一个问题：

```text
怎么让模型在训练数据上越学越好？
```

但真正做模型时，还有一个更重要的问题：

```text
模型是不是只把训练数据背下来了？
还是它真的学到了可以用在新数据上的规律？
```

这一节就是正则化之前的桥梁课：先学会看训练曲线，理解泛化能力。

## 1. 训练集不是最终目标

训练时，我们会最小化训练损失：

$$
\min_{\theta}\mathcal{L}_{train}
$$

这个式子只是说：找到一组参数 $\theta$，让模型在训练集上的错误尽量小。

但训练集不是最终目标。

真正目标是：

```text
模型在没见过的新数据上也能表现好。
```

如果一个学生只背熟了练习册原题，考试换个数字就不会了，那不能叫真正学会。

模型也是一样。

## 2. 为什么要分训练集、验证集、测试集

通常我们会把数据拆成三部分：

| 数据集 | 用来做什么 | 类比 |
| --- | --- | --- |
| 训练集 | 用来更新参数 | 练习题 |
| 验证集 | 用来观察模型是否泛化 | 模拟考试 |
| 测试集 | 最后评估模型 | 正式考试 |

训练集参与参数更新。

验证集不直接更新参数，它的作用是帮我们判断：模型是不是开始只适应训练集了。

测试集最好只在最后使用，因为它代表最终评估。

## 3. 什么是泛化能力

泛化能力就是模型处理新数据的能力。

训练损失小，只能说明模型会做训练题。

验证损失小，才更能说明模型学到了比较稳定的规律。

可以把泛化能力理解成：

```text
模型从训练数据中学到的东西，能不能迁移到没见过的数据上。
```

所以我们不能只盯着：

$$
\mathcal{L}_{train}
$$

还要看：

$$
\mathcal{L}_{val}
$$

这两个损失之间的关系，才是诊断训练状态的关键。

## 4. 什么是泛化差距

泛化差距可以简单理解成：训练表现和验证表现之间的差距。

如果用损失表示，可以写成：

$$
\text{generalization gap}=\mathcal{L}_{val}-\mathcal{L}_{train}
$$

这个公式不用想复杂。

它就是在问：

```text
模型在没见过的数据上，比在训练数据上差多少？
```

如果 $\mathcal{L}_{train}$ 很低，但 $\mathcal{L}_{val}$ 明显更高，说明模型可能太适应训练集了。

这就是后面要讲的过拟合。

## 5. 第一种曲线：正常学习

比较理想的训练过程大概是：

```text
训练 loss 下降
验证 loss 也下降
两者差距不大
```

用符号表示就是：

$$
\mathcal{L}_{train}\downarrow,\qquad \mathcal{L}_{val}\downarrow
$$

这说明模型不仅在训练集上变好，在新数据上也变好。

这时候可以说：模型学到了一些有用规律。

## 6. 第二种曲线：欠拟合

欠拟合是指模型连训练集都学不好。

典型现象是：

$$
\mathcal{L}_{train}\text{ 很高},\qquad \mathcal{L}_{val}\text{ 也很高}
$$

这说明问题不是模型背题背过头，而是它连基本规律都没学到。

常见原因包括：

```text
模型太简单
训练轮数太少
学习率设置不合适
特征或数据本身有问题
```

欠拟合像是学生连练习题都做不对。

## 7. 第三种曲线：过拟合

过拟合是指模型在训练集上越来越好，但在验证集上变差。

典型现象是：

$$
\mathcal{L}_{train}\downarrow,\qquad \mathcal{L}_{val}\uparrow
$$

这说明模型确实把训练集学得越来越熟。

但问题是：它学进去的不一定都是通用规律，可能还包括训练集里的偶然细节、噪声、特殊样本。

过拟合像是学生把练习册答案背熟了，但题目稍微换一下就不会。

## 8. 为什么模型会过拟合

过拟合不是因为模型太笨，而常常是因为模型太有能力。

神经网络参数很多、表达能力强，它不仅能学到主要规律，也有能力把训练集里的细节记住。

过拟合通常来自三件事：

```text
模型太复杂
数据太少
训练太久
```

如果数据足够多，模型更容易看到各种情况，就不容易把某几个样本当成普遍规律。

如果模型太复杂，它就有更多空间去记住训练集细节。

如果训练太久，模型可能先学大规律，后面慢慢开始学噪声。

## 9. 决策边界角度理解过拟合

还记得我们前面讲过决策边界吗？

决策边界就是模型把不同类别分开的那条线、那个面，或者更高维空间里的分割边界。

如果模型比较简单，决策边界可能比较平滑。

如果模型很复杂，它可能为了照顾训练集里的每一个特殊点，把边界扭得很厉害。

这时候训练集上可能分得特别好。

但新样本一来，因为边界太贴合训练样本的偶然形状，反而容易判断错。

所以过拟合可以理解成：

```text
决策边界过度贴合训练集细节，而不是抓住真正稳定的分界规律。
```

## 10. 正则化为什么会出现

现在正则化就不是凭空出现了。

如果问题是模型太容易把训练集细节记住，那我们就需要给模型一点限制。

正则化的核心思想是：

```text
允许模型学习训练集，但不要让它用过于复杂、过于极端的方式去拟合训练集。
```

普通训练只关心：

$$
\mathcal{L}_{train}
$$

正则化会在目标里加一个约束：

$$
\mathcal{L}_{total}=\mathcal{L}_{train}+\lambda\Omega(\theta)
$$

$\Omega(\theta)$ 表示模型复杂度惩罚。

$\lambda$ 表示惩罚强度。

这句话翻译一下就是：

```text
你可以降低训练错误，但别把模型搞得太复杂。
```

## 11. 看曲线时先问什么

以后你看到训练曲线，先不要急着加技巧。

先问两个问题：

```text
训练 loss 高不高？
验证 loss 和训练 loss 差距大不大？
```

如果训练 loss 高，验证 loss 也高，优先怀疑欠拟合。

如果训练 loss 低，验证 loss 高，优先怀疑过拟合。

如果两个 loss 都在下降，说明训练还在正常学习。

这个判断比死记技巧重要得多。

## 12. 欠拟合时通常怎么处理

欠拟合说明模型还没学够。

常见处理思路是：

```text
增加模型表达能力
训练更久
调整学习率
检查数据预处理
减少过强的正则化
```

注意最后一点：正则化太强也可能导致欠拟合。

因为你限制模型限制得太厉害，它连训练集都学不好。

## 13. 过拟合时通常怎么处理

过拟合说明模型对训练集太熟，对新数据不够稳。

常见处理思路是：

```text
增加数据
做数据增强
使用权重衰减
使用 Dropout
使用早停
降低模型复杂度
```

这些方法看起来很多，但都围绕同一件事：

```text
减少模型对训练集偶然细节的依赖。
```

## 14. 训练准确率和验证准确率也能这样看

如果是分类任务，除了 loss，还经常看 accuracy。

过拟合时可能出现：

$$
\text{Acc}_{train}\uparrow,\qquad \text{Acc}_{val}\downarrow
$$

或者：

$$
\text{Acc}_{train}\text{ 很高},\qquad \text{Acc}_{val}\text{ 明显更低}
$$

含义一样：模型在训练题上越来越熟，但新题做不好。

所以判断过拟合可以看 loss，也可以看 accuracy。

只是 loss 通常更细腻，accuracy 有时候变化不明显。

## 15. 本节总结

这一节的逻辑链是：

```text
训练集 loss 低不等于模型真的好
-> 真正目标是泛化能力
-> 验证集用来观察模型在新数据上的表现
-> 训练损失和验证损失的差距叫泛化差距
-> 训练 loss 高、验证 loss 高：可能欠拟合
-> 训练 loss 低、验证 loss 高：可能过拟合
-> 正则化是为了缓解过拟合，让模型不要太依赖训练集细节
```

先记住一句话：

```text
训练集看学习能力，验证集看泛化能力；两条曲线一起看，才知道模型到底怎么了。
```

下一节再正式进入正则化方法：权重衰减、Dropout、早停、数据增强。